# Klasifikasi 15 kelas — YOLOv8 di Kaggle (`ryanbadai/clothes-dataset`)

Notebook ini melatih **classifier Gayaku** dengan **Ultralytics YOLOv8** (`yolov8n-cls.pt`, 224 px) di Kaggle, memakai dataset `ryanbadai/clothes-dataset` yang kamu lampirkan sebagai input. Notebook ini **berdiri sendiri** (self-contained): ia tidak meng-clone repo, jadi tabel label, pembagian data, dan semua pemeriksaan kontrak ditulis di dalamnya.

### Menyiapkan sesi Kaggle

1. Panel kanan → **Accelerator → GPU T4** (atau P100).
2. Panel kanan → **Internet → On**. Kalau internet mati, `pip install ultralytics` **dan** unduhan bobot `yolov8n-cls.pt` gagal; langkah 1 akan memberitahumu kalau itu terjadi.
3. **Add Input → `ryanbadai/clothes-dataset`**, lalu jalankan langkah 2 — kalau path-nya belum ada, langkah 2 berhenti dengan pesan yang menyuruhmu melampirkan dataset itu.

### Batas Kaggle yang perlu kamu tahu

* Kuota GPU gratis ±**30 jam/pekan**. Habis itu, sesi GPU tidak bisa dimulai lagi sampai pekan berikutnya.
* Satu sesi maksimal **12 jam**.
* `/kaggle/working` otomatis disimpan (maks **20 GB**) saat notebook di-*commit*, dan itulah yang jadi output notebook-mu.
* **Save & Run All (Commit)** berjalan di server, jadi tab boleh ditutup — berbeda dari sesi interaktif yang mati kalau tab ditutup. Tapi setiap commit **memakai kuota GPU**. Karena itu: iterasi dengan sesi interaktif (Run All biasa), dan commit **sekali** di akhir kalau kamu memang butuh menjalankannya tanpa membuka tab.
* Dataset input bersifat *read-only*. Notebook ini menaruh pohon data (symlink) dan hasil latihan di `/kaggle/temp` supaya **tidak** ikut tersimpan ke output commit — lihat langkah 3.

### Kontrak aplikasi (tidak boleh berubah)

| | |
| --- | --- |
| input | float32 (`float16` ditoleransi), NHWC `[1,224,224,3]` atau NCHW `[1,3,224,224]`, RGB `[0,1]`; normalisasi apa pun harus ikut ter-*bake* di dalam model |
| output | satu tensor `[1,N]` float32, `N` = jumlah label di `classes.json`, nilainya distribusi probabilitas (jumlah ≈ 1) |
| urutan kelas | urutan `labels` di `classes.json` — di notebook ini = `model.names`, yaitu urutan abjad nama folder kelas |

Yang menegakkan kontrak itu: `flutter_app/lib/logic/scan_pipeline.dart::checkContracts` saat model dimuat (aplikasi melempar pesan Indonesia kalau jumlah kelasnya tidak cocok), dan `ml_workspace/scripts/gen_dart_labels.py --model` saat codegen.

### Bacaan terkait

* `docs/adr/0007-carousell-clothes-dataset.md` — dataset ini: 15 kelas, ±500 foto/kelas, tanpa split, plus catatan noise label.
* `docs/adr/0006-yolov8-classifier-agpl.md` — pilihan YOLOv8 dan konsekuensi lisensi **AGPL-3.0**.
* `docs/adr/0005-classify-the-cutout.md` — kenapa foto satu pakaian per frame itu syarat, bukan preferensi.
* `ml_workspace/yolo/README.md` — langkah integrasi setelah zip ini kamu unduh.

## 1. Runtime, GPU, dan instalasi

Sel pertama melakukan tiga hal: memastikan **GPU terlihat**, memasang **ultralytics**, dan mencetak versi Python/PyTorch/ultralytics.

Kenapa versi dicetak: ultralytics mengubah API antar rilis, jadi angka versi adalah cara membedakan "bug kita" dari "perubahan mereka" ketika hasilnya berbeda dari yang tertulis di `report.md` orang lain.

Kalau CUDA tidak terlihat, accelerator belum dipasang: panel kanan → **Accelerator: GPU T4**. Notebook tetap jalan di CPU, tapi 30 epoch × ±6.000 foto di CPU bisa berjam-jam, dan itu bukan cara memakai kuota mingguanmu.

Kalau `pip install` gagal dengan error jaringan (bukan error paket), **Internet** masih **Off** di panel kanan — nyalakan lalu ulangi sel ini. Tanpa internet, bobot awal `yolov8n-cls.pt` juga tidak bisa diunduh, jadi latihan tidak bisa mulai.

In [ ]:
# Langkah 1/8 — cek runtime dan pasang ultralytics.
# Kalau sel ini error "Could not find a version that satisfies the requirement" / koneksi macet:
# panel kanan -> Internet -> On, lalu jalankan ulang sel ini.
!pip install -q ultralytics

import importlib.util
import sys

import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")

GPU = torch.cuda.is_available()
if GPU:
    DEVICE = 0
    print(f"cuda          {torch.version.cuda}")
    print(f"gpu           {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("cuda          TIDAK ADA")
    print("              panel kanan -> Accelerator -> GPU T4, lalu jalankan ulang dari sel ini")

# Langkah 6 memverifikasi .tflite dengan runtime yang terpisah dari exporter-nya.
# Kaggle sudah menyediakan TensorFlow; ai_edge_litert ikut terpasang bersama litert-torch saat export.
for module in ("tensorflow", "ai_edge_litert"):
    print(f"{module:<14}{'tersedia' if importlib.util.find_spec(module) else 'belum terpasang'}")

## 2. Cari dataset dan periksa isinya

Dataset ini adalah **ImageFolder datar**: `Clothes_Dataset/<Kelas>/*.jpg`, 15 folder kelas berbahasa Indonesia, ±500 foto per kelas (±7.500 foto, 1,48 GB), dan **tidak ada folder train/validation/test**. Karena itu langkah 3 yang membuat pembagiannya sendiri.

Path input di Kaggle bersifat *read-only* dan selalu berbentuk `/kaggle/input/<slug>/…`, jadi sel ini mencari `Clothes_Dataset` di dua tempat: path yang didokumentasikan (`/kaggle/input/clothes-dataset/Clothes_Dataset`) lalu — kalau tidak ada — berkeliling `/kaggle/input` untuk berjaga-jaga kalau slug-nya berbeda tipis. Kalau tetap tidak ketemu, sel berhenti dengan instruksi melampirkan `ryanbadai/clothes-dataset` lewat **Add Input**; itu lebih menolong daripada traceback `FileNotFoundError`.

Sel ini juga membandingkan **nama folder** dengan 15 label yang diharapkan repo. Nama folder itulah yang menjadi nama kelas di model, jadi kalau dataset di Kaggle diubah (folder ditambah/diganti nama), kita ingin tahu **di sini**, bukan setelah `classes.json` bertentangan dengan `ml_workspace/labels.json`.

### Tabel label — harus sama dengan `ml_workspace/labels.json`

| label (nama folder) | Category | Nama tampilan |
| --- | --- | --- |
| `Blazer` | Outerwear | Blazer |
| `Celana_Panjang` | Bottoms | Celana Panjang |
| `Celana_Pendek` | Bottoms | Celana Pendek |
| `Gaun` | Tops | Gaun |
| `Hoodie` | Tops | Hoodie |
| `Jaket` | Outerwear | Jaket |
| `Jaket_Denim` | Outerwear | Jaket Denim |
| `Jaket_Olahraga` | Outerwear | Jaket Olahraga |
| `Jeans` | Bottoms | Jeans |
| `Kaos` | Tops | Kaos |
| `Kemeja` | Tops | Kemeja |
| `Mantel` | Outerwear | Mantel |
| `Polo` | Tops | Kaos Polo |
| `Rok` | Bottoms | Rok |
| `Sweter` | Outerwear | Sweter |

Lima Category dari `CONTEXT.md` hanya punya lima slot (Tops, Bottoms, Outerwear, Shoes, Accessories), jadi dataset ini memakai tiga di antaranya: **Tops 5, Bottoms 4, Outerwear 6**. Tidak ada kelas Shoes maupun Accessories — artinya model ini **tidak akan mengenali sepatu atau tas**; itu keterbatasan daftar kelas, bukan kegagalan model, dan sudah tercatat di `docs/adr/0007-carousell-clothes-dataset.md`.

Dua hal yang harus kamu tahu tentang tabel di atas:

* `Gaun` (dress) dan `Hoodie` masuk **Tops**: pakaian satu potong tidak punya slot jujurnya di lima Category, jadi ia jatuh ke tetangga terdekat (`Hoodie` juga bukan Outerwear menurut `labels.json` — ikuti file itu, bukan perasaanmu).
* `Jeans`, `Polo`, `Hoodie` dan `Blazer` **sengaja memakai ulang** Category dan nama tampilan dari entri dataset lain di `ml_workspace/labels.json`. `gen_dart_labels.py` menolak dua dataset yang labelnya sama tapi Category/nama berbeda, jadi tabel ini tidak boleh "diperbaiki" sepihak di sini: kalau kamu mengubahnya, ubah `ml_workspace/labels.json` juga, di commit yang sama.

In [ ]:
# Langkah 2/8 — temukan dataset, periksa jumlah per kelas, dan cocokkan nama foldernya.
from pathlib import Path

DOCUMENTED_ROOT = Path("/kaggle/input/clothes-dataset/Clothes_Dataset")

# Tabel label: disalin apa adanya dari 15 baris dataset 'ryanbadai/clothes-dataset' di
# ml_workspace/labels.json. Nama folder di dataset = label di sini, dan urutan Category/Nama
# harus tetap sama dengan file itu: gen_dart_labels.py menolak dua dataset yang labelnya sama
# tapi Category atau nama tampilannya berbeda (Jeans, Polo, Hoodie, Blazer dipakai bersama).
LABEL_TABLE = [
    {"label": 'Blazer', "category": 'Outerwear', "name": 'Blazer'},
    {"label": 'Celana_Panjang', "category": 'Bottoms', "name": 'Celana Panjang'},
    {"label": 'Celana_Pendek', "category": 'Bottoms', "name": 'Celana Pendek'},
    {"label": 'Gaun', "category": 'Tops', "name": 'Gaun'},
    {"label": 'Hoodie', "category": 'Tops', "name": 'Hoodie'},
    {"label": 'Jaket', "category": 'Outerwear', "name": 'Jaket'},
    {"label": 'Jaket_Denim', "category": 'Outerwear', "name": 'Jaket Denim'},
    {"label": 'Jaket_Olahraga', "category": 'Outerwear', "name": 'Jaket Olahraga'},
    {"label": 'Jeans', "category": 'Bottoms', "name": 'Jeans'},
    {"label": 'Kaos', "category": 'Tops', "name": 'Kaos'},
    {"label": 'Kemeja', "category": 'Tops', "name": 'Kemeja'},
    {"label": 'Mantel', "category": 'Outerwear', "name": 'Mantel'},
    {"label": 'Polo', "category": 'Tops', "name": 'Kaos Polo'},
    {"label": 'Rok', "category": 'Bottoms', "name": 'Rok'},
    {"label": 'Sweter', "category": 'Outerwear', "name": 'Sweter'},
]

EXPECTED_LABELS = [row["label"] for row in LABEL_TABLE]
METADATA = {row["label"]: row for row in LABEL_TABLE}
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png")


def find_dataset_root():
    """Path yang didokumentasikan dulu, lalu berkeliling /kaggle/input."""
    if DOCUMENTED_ROOT.is_dir():
        return DOCUMENTED_ROOT
    for candidate in sorted(Path("/kaggle/input").rglob("Clothes_Dataset")):
        if candidate.is_dir():
            print(f"catatan: {DOCUMENTED_ROOT} tidak ada, memakai {candidate}")
            return candidate
    return None


DATASET_ROOT = find_dataset_root()
if DATASET_ROOT is None:
    raise SystemExit(
        "Dataset tidak ditemukan.\n"
        "  1. Panel kanan -> Add Input -> cari 'ryanbadai/clothes-dataset' -> tambahkan.\n"
        "  2. Jalankan ulang sel ini (dataset baru muncul setelah sesi direstart/ditambahkan).\n"
        f"  Path yang dicari: {DOCUMENTED_ROOT}"
    )

IMAGES = {
    folder.name: sorted(path for path in folder.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    for folder in sorted(DATASET_ROOT.iterdir())
    if folder.is_dir()
}
if sorted(IMAGES) != sorted(EXPECTED_LABELS):
    tambahan = sorted(set(IMAGES) - set(EXPECTED_LABELS))
    hilang = sorted(set(EXPECTED_LABELS) - set(IMAGES))
    raise SystemExit(
        "Nama folder kelas tidak sama dengan 15 label yang diharapkan.\n"
        f"  tidak diharapkan: {tambahan}\n"
        f"  tidak ditemukan  : {hilang}\n"
        "  Perbaiki ml_workspace/labels.json + tabel di atas, atau pakai snapshot dataset yang benar."
    )

print(f"dataset       {DATASET_ROOT}")
print(f"{'kelas':<18}{'Category':<12}{'Nama':<12}{'jumlah':>8}")
for label, photos in IMAGES.items():
    print(f"{label:<18}{METADATA[label]['category']:<12}{METADATA[label]['name']:<12}{len(photos):>8}")
print(f"{'TOTAL':<42}{sum(len(photos) for photos in IMAGES.values()):>8}")

terlalu_tipis = {label: len(photos) for label, photos in IMAGES.items() if len(photos) < 100}
if terlalu_tipis:
    raise SystemExit(f"kelas dengan <100 foto tidak bisa dibagi 80/20 dengan sehat: {terlalu_tipis}")

## 3. Bagi data 80/20 secara stratified, lewat symlink

Dataset ini tidak punya split, jadi kita membuatnya: **80 % train, 20 % validation**, diacak **per kelas** (stratified) dengan seed tetap supaya hasil pembagian bisa diulang kata per kata. Validation itulah satu-satunya angka di notebook ini yang tidak "berbohong halus" — model tidak pernah melihatnya saat latihan maupun saat memilih epoch terbaik.

Pohon data dibangun dengan **symlink**, bukan salinan: 1,48 GB foto tidak perlu digandakan, dan berkasnya tetap milik folder input yang read-only. Struktur hasilnya adalah yang diminta ultralytics untuk klasifikasi: `dataset/train/<Kelas>/*.jpg` + `dataset/val/<Kelas>/*.jpg` — dan **nama folder adalah nama kelas**, sehingga `model.names` keluar berurutan abjad. Itu sebabnya `classes.json` di langkah 7 ditulis mengikuti `model.names`, bukan urutan tabel di atas.

Symlink dan hasil latihan ditaruh di **`/kaggle/temp`**, bukan `/kaggle/working`: `/kaggle/working` ikut tersimpan saat commit (maks 20 GB), dan ribuan symlink plus checkpoint `best.pt` bukan sesuatu yang ingin kamu simpan sebagai output notebook. Kalau `/kaggle/temp` ternyata tidak bisa ditulis di lingkunganmu, sel ini otomatis jatuh ke `/kaggle/working/temp`.

In [ ]:
# Langkah 3/8 — split stratified 80/20 dan bangun ImageFolder lewat symlink.
import json
import os
import random
import shutil
from pathlib import Path

SEED = 42          # seed pembagian data
VAL_FRACTION = 0.2  # 20 % jadi data uji internal; baseline publik memakai 40 %

# /kaggle/temp tidak ikut tersimpan saat commit; /kaggle/working ikut (maks 20 GB).
WORK_DIR = Path("/kaggle/temp") if os.access("/kaggle/temp", os.W_OK) else Path("/kaggle/working/temp")
WORK_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = WORK_DIR / "runs"
DATASET_DIR = WORK_DIR / "dataset"
ARTIFACTS_DIR = Path("/kaggle/working/artifacts")   # inilah yang ikut tersimpan
for path in (RUNS_DIR, DATASET_DIR, ARTIFACTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print(f"kerja         {WORK_DIR}")
print(f"artefak       {ARTIFACTS_DIR}")

BUILD_MARKER = DATASET_DIR / ".build.json"
marker = {}
if BUILD_MARKER.is_file():
    try:
        marker = json.loads(BUILD_MARKER.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        marker = {}

if marker.get("seed") == SEED and marker.get("val_fraction") == VAL_FRACTION:
    print(f"memakai ulang {DATASET_DIR} (hapus foldernya kalau ingin membangun ulang)")
else:
    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    rng = random.Random(SEED)
    for label, photos in IMAGES.items():
        shuffled = list(photos)
        rng.shuffle(shuffled)
        n_val = max(1, round(len(shuffled) * VAL_FRACTION))
        for split, group in (("train", shuffled[n_val:]), ("val", shuffled[:n_val])):
            target = DATASET_DIR / split / label
            target.mkdir(parents=True, exist_ok=True)
            for photo in group:
                os.symlink(photo, target / photo.name)
    BUILD_MARKER.write_text(
        json.dumps({"seed": SEED, "val_fraction": VAL_FRACTION, "source": str(DATASET_ROOT)}, indent=2),
        encoding="utf-8",
    )
    print(f"pohon data dibangun di {DATASET_DIR} (symlink, tanpa menyalin byte)")

COUNTS = {
    label: {split: len(list((DATASET_DIR / split / label).iterdir())) for split in ("train", "val")}
    for label in IMAGES
}
print(f"{'kelas':<18}{'train':>7}{'val':>6}")
for label, counts in COUNTS.items():
    print(f"{label:<18}{counts['train']:>7}{counts['val']:>6}")
print(f"{'TOTAL':<18}{sum(c['train'] for c in COUNTS.values()):>7}{sum(c['val'] for c in COUNTS.values()):>6}")

CLASSES = sorted(folder.name for folder in (DATASET_DIR / "train").iterdir() if folder.is_dir())
if CLASSES != sorted(EXPECTED_LABELS):
    raise SystemExit(f"daftar kelas di pohon data tidak sesuai: {CLASSES}")
print(f"{len(CLASSES)} kelas, urutan model.names: {CLASSES}")

## 4. Latihan

`yolov8n-cls.pt` adalah checkpoint **klasifikasi nano** (beberapa MB, sudah terlatih di ImageNet): ±6.000 foto dengan 30 epoch selesai dalam puluhan menit di GPU Kaggle. Ini klasifikasi, bukan deteksi — tidak ada bounding box dan tidak ada NMS, jadi graf hasilnya hanya punya satu input gambar dan satu output vektor probabilitas.

Kenop yang perlu kamu tahu, semuanya variabel di awal sel:

* `EPOCHS` (30), `BATCH` (64 cukup untuk GPU Kaggle pada 224 px), `SEED` (42), `IMGSZ` (224 — kontrak aplikasi);
* `PATIENCE = 10`: berhenti lebih awal kalau validation tidak membaik, daripada membakar kuota 100 epoch;
* `workers=2`: mesin Kaggle biasanya memberi 2–4 vCPU, jadi worker besar hanya menambah overhead;
* `project=RUNS_DIR`, `name=RUN_NAME`: hasilnya masuk ke `/kaggle/temp/runs/…`, bukan `/kaggle/working`, supaya checkpoint tidak ikut jadi output commit.

`best.pt` (epoch dengan validation top-1 terbaik) adalah yang dipakai semua langkah berikutnya. Baris terakhir sel memastikan urutan kelas model sama dengan urutan folder di langkah 3 — itu yang membuat `classes.json` di langkah 7 punya arti.

In [ ]:
# Langkah 4/8 — latih yolov8n-cls pada 224 px.
from pathlib import Path

from ultralytics import YOLO

MODEL = "yolov8n-cls.pt"   # coba "yolo11n-cls.pt" atau "yolov8s-cls.pt" untuk backbone yang lebih besar
EPOCHS = 30
BATCH = 64
SEED_TRAIN = 42            # seed latihan (SEED di langkah 3 mengatur pembagian data)
IMGSZ = 224                # kontrak aplikasi
PATIENCE = 10
RUN_NAME = "yolov8n-cls-clothes-224"

model = YOLO(MODEL)
final_metrics = model.train(
    data=str(DATASET_DIR),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED_TRAIN,
    patience=PATIENCE,
    device=DEVICE,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,   # jalankan ulang berarti menimpa run ini, bukan membuat run2, run3 ...
    pretrained=True,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
CLASS_NAMES = [model.names[index] for index in range(len(model.names))]

print(f"run           {RUN_DIR}")
print(f"bobot terbaik {BEST_WEIGHTS} ({BEST_WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print(f"metrik akhir  {final_metrics}")
print(f"{len(CLASS_NAMES)} kelas dalam urutan model: {CLASS_NAMES}")
if CLASS_NAMES != CLASSES:
    raise SystemExit("urutan kelas model berbeda dari folder dataset — model ini jangan dipakai")

## 5. Evaluasi: top-1, top-5, `classification_report`, confusion matrix

`split="val"` adalah 20 % yang tidak pernah dipakai berlatih dan tidak dipakai memilih epoch terbaik, jadi inilah angka yang layak dilaporkan. `metrics.top1` / `metrics.top5` berasal dari ultralytics; confusion matrix-nya juga.

Untuk **`classification_report`** (precision/recall/F1 per kelas) kita perlu daftar prediksi per foto, bukan hanya matriksnya — karena itu sel ini melakukan satu pass tambahan `model.predict` atas folder `val` dan mengambil label asli dari **nama folder induk** tiap hasil, bukan dari urutan file. Pass ini murah (hanya ±1.200 foto) dan sekaligus jadi pemeriksaan silang: matriks dari pass prediksi dibandingkan dengan matriks ultralytics, dan sel mencetak berapa sel yang berbeda (idealnya 0 — perbedaan kecil bisa muncul dari pembulatan/urutan pipeline, bukan dari model).

`macro-F1` itulah angka yang bisa dibandingkan dengan baseline publik (ViT-base: **78,47 % akurasi / 0,7803 macro-F1**, 30 epoch, split 40 %), dengan catatan penting soal split di langkah 7.

In [ ]:
# Langkah 5/8 — evaluasi pada split val.
import csv as csv_module
import shutil as shutil_module

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix as sklearn_confusion_matrix
from ultralytics import YOLO

VAL_SPLIT = "val"

trained = YOLO(str(BEST_WEIGHTS))
metrics = trained.val(
    data=str(DATASET_DIR),
    split=VAL_SPLIT,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    plots=True,
    project=str(RUNS_DIR),
    name=f"val-{VAL_SPLIT}",
    exist_ok=True,
)
TOP1 = float(metrics.top1)
TOP5 = float(metrics.top5)
print(f"top-1         {TOP1:.4f}")
print(f"top-5         {TOP5:.4f}")

confusion = metrics.confusion_matrix
CONFUSION_NAMES = [confusion.names[index] for index in range(confusion.nc)]
ULTRA_MATRIX = np.asarray(confusion.matrix)   # [predicted][true]

# Pass kedua: label per foto, untuk classification_report.
results = trained.predict(source=str(DATASET_DIR / VAL_SPLIT), imgsz=IMGSZ, batch=BATCH, device=DEVICE, verbose=False)
y_true = [Path(result.path).parent.name for result in results]
y_pred = [CLASSES[int(result.probs.top1)] for result in results]
print(f"prediksi      {len(results)} foto val")

REPORT_TEXT = classification_report(
    y_true, y_pred, labels=CLASSES, target_names=CLASSES, digits=4, zero_division=0
)
REPORT_DICT = classification_report(
    y_true, y_pred, labels=CLASSES, target_names=CLASSES, digits=4, zero_division=0, output_dict=True
)
MACRO_F1 = float(REPORT_DICT["macro avg"]["f1-score"])
ACCURACY = float(REPORT_DICT["accuracy"])
print(REPORT_TEXT)
print(f"akurasi       {ACCURACY:.4f}")
print(f"macro-F1      {MACRO_F1:.4f}")

SKLEARN_MATRIX = sklearn_confusion_matrix(y_true, y_pred, labels=CLASSES)  # [true][predicted]
beda = int((ULTRA_MATRIX != SKLEARN_MATRIX.T).sum())
print(f"silang cek    {beda} sel berbeda antara matriks ultralytics dan scikit-learn (idealnya 0)")

with (ARTIFACTS_DIR / "classification_report.txt").open("w", encoding="utf-8") as handle:
    handle.write(REPORT_TEXT)
with (ARTIFACTS_DIR / "classification_report.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv_module.writer(handle)
    writer.writerow(("label", "precision", "recall", "f1-score", "support"))
    for label in CLASSES:
        row = REPORT_DICT[label]
        writer.writerow((label, f"{row['precision']:.4f}", f"{row['recall']:.4f}", f"{row['f1-score']:.4f}", int(row["support"])))
    writer.writerow(
        ("macro avg", f"{MACRO_F1:.4f}", f"{MACRO_F1:.4f}", f"{MACRO_F1:.4f}", sum(int(REPORT_DICT[l]["support"]) for l in CLASSES))
    )

SUPPORT = SKLEARN_MATRIX.sum(axis=0)
CORRECT = np.diag(SKLEARN_MATRIX)
RECALL = {label: float(CORRECT[index] / max(1.0, SUPPORT[index])) for index, label in enumerate(CLASSES)}

with (ARTIFACTS_DIR / "per_class_accuracy.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv_module.writer(handle)
    writer.writerow(("label", "support", "correct", "recall"))
    for index, label in enumerate(CLASSES):
        writer.writerow((label, int(SUPPORT[index]), int(CORRECT[index]), f"{RECALL[label]:.4f}"))

with (ARTIFACTS_DIR / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv_module.writer(handle)
    writer.writerow(("predicted \\ true", *CLASSES))
    for index, label in enumerate(CLASSES):
        writer.writerow((label, *[int(value) for value in ULTRA_MATRIX[index]]))

for png in sorted(Path(metrics.save_dir).glob("confusion_matrix*.png")):
    shutil_module.copy2(png, ARTIFACTS_DIR / png.name)
    print(f"grafik        {ARTIFACTS_DIR / png.name}")

## 6. Export ke `.tflite`, lalu verifikasi

**Bagian 1** mengekspor graf: `format="litert"` adalah nama sekarang untuk ekspor LiteRT (menghasilkan `.tflite`); `format="tflite"` masih jalan tapi sudah usang dan hanya diteruskan ke nama baru itu dengan peringatan. Untuk klasifikasi tidak ada NMS maupun decoding box di dalam graf. Ekspor dijalankan di CPU: tracing tidak butuh GPU, dan model sekecil ini selesai jauh di bawah satu menit.

**Bagian 2** yang menentukan file ini boleh dipakai atau tidak, dan sengaja paranoid — model yang salah tapi *tetap bisa dimuat* lebih berbahaya daripada model yang gagal dimuat:

* Pemeriksaan memakai **runtime independen** (`tf.lite.Interpreter`, atau `ai_edge_litert`), bukan exporter yang menulis filenya. Yang diperiksa persis sama dengan yang ditegakkan `scan_pipeline.dart::checkContracts`: input float32, batch 1, `[1,224,224,3]` **atau** `[1,3,224,224]` (sel mencetak yang mana), output `[1,15]` float32 dengan jumlah ≈ 1.
* Kalau output ternyata *logits* (jumlah ≠ 1), sel mengekspor ulang dengan `nn.Softmax` di ujungnya lalu memeriksa lagi. Aplikasi sebenarnya sudah menormalkan logits sendiri (`probabilitiesFrom`), tapi kontrak repo menyebut "distribusi probabilitas", dan `ml_workspace/scripts/classify_assets.py` membaca skornya sebagai keyakinan — jadi lebih baik benar di sumbernya.
* Lalu **gerbang fidélitas**: ≥100 foto held-out dilewatkan ke model PyTorch **dan** ke `.tflite` dengan **piksel yang sama** (resize sisi pendek + center crop, RGB `[0,1]`), lalu dicetak berapa argmax yang setuju. Gerbangnya sama dengan yang dipakai `ml_workspace/bootstrap/export_tflite.py tflite`: akurasi `.tflite` ≥ 0,50 dan tidak lebih dari 0,10 di bawah PyTorch.

In [ ]:
# Langkah 6/8 (bagian 1) — ekspor ke LiteRT (.tflite).
import hashlib

from ultralytics import YOLO

TFLITE_PATH = ARTIFACTS_DIR / "garment_classifier.tflite"
EXPORT_STEPS = []

trained = YOLO(str(BEST_WEIGHTS))
exported_path = Path(trained.export(format="litert", imgsz=IMGSZ, device="cpu"))
shutil.copy2(exported_path, TFLITE_PATH)
EXPORT_STEPS.append(f"YOLO('{BEST_WEIGHTS}').export(format='litert', imgsz={IMGSZ}, device='cpu')")

print(f"diekspor     {exported_path}")
print(f"disalin ke   {TFLITE_PATH} ({TFLITE_PATH.stat().st_size / 1e6:.2f} MB)")
print(f"sha256       {hashlib.sha256(TFLITE_PATH.read_bytes()).hexdigest()}")

In [ ]:
# Langkah 6/8 (bagian 2) — verifikasi kontrak aplikasi, lalu bandingkan dengan PyTorch.
import importlib.util
import math
import shutil
from collections import namedtuple

import numpy as np
import torch
from PIL import Image

Contract = namedtuple(
    "Contract", "runtime interpreter in_index in_shape in_dtype out_index out_shape out_dtype layout"
)


def load_interpreter(path):
    """Runtime TFLite yang independen dari exporter yang menulis filenya."""
    if importlib.util.find_spec("tensorflow"):
        import tensorflow as tf

        return tf.lite.Interpreter(model_path=str(path)), f"tensorflow {tf.__version__} tf.lite.Interpreter"
    from ai_edge_litert.interpreter import Interpreter

    return Interpreter(model_path=str(path)), "ai_edge_litert.interpreter.Interpreter"


def inspect_model(path):
    """Baca IO tensor yang akan dilihat aplikasi, sekaligus layout-nya (NHWC atau NCHW)."""
    interpreter, runtime = load_interpreter(path)
    interpreter.allocate_tensors()
    inputs = interpreter.get_input_details()[0]
    outputs = interpreter.get_output_details()[0]
    shape = tuple(int(value) for value in inputs["shape"])
    if shape[1:] == (IMGSZ, IMGSZ, 3):
        layout = "NHWC"
    elif shape[1:] == (3, IMGSZ, IMGSZ):
        layout = "NCHW"
    else:
        raise SystemExit(f"bentuk input {shape} tidak dikenal: aplikasi menerima [1,224,224,3] atau [1,3,224,224]")
    return Contract(
        runtime,
        interpreter,
        inputs["index"],
        shape,
        inputs["dtype"],
        outputs["index"],
        tuple(int(value) for value in outputs["shape"]),
        outputs["dtype"],
        layout,
    )


def assert_app_contract(contract, class_count):
    """Persis yang ditolak flutter_app/lib/logic/scan_pipeline.dart."""
    assert contract.in_dtype == np.float32, f"dtype input {contract.in_dtype.__name__}: aplikasi mengirim float32"
    assert contract.out_dtype == np.float32, f"dtype output {contract.out_dtype.__name__}: aplikasi membaca float32"
    assert contract.in_shape[0] == 1 and contract.out_shape[0] == 1, "kedua tensor harus batch 1"
    assert contract.out_shape[1] == class_count, (
        f"model memberi {contract.out_shape[1]} skor tapi classes.json akan berisi {class_count}: "
        "aplikasi melempar pesan Indonesia yang menyebut ketidakcocokan ini"
    )


def predict(contract, pixels_hwc):
    """[H,W,3] float32 RGB [0,1] -> output model [N] float32."""
    array = pixels_hwc[None] if contract.layout == "NHWC" else np.transpose(pixels_hwc, (2, 0, 1))[None]
    contract.interpreter.set_tensor(contract.in_index, array.astype(np.float32))
    contract.interpreter.run()
    return contract.interpreter.get_tensor(contract.out_index)[0]


def load_pixels(path, size=IMGSZ):
    """Piksel identik untuk kedua model: RGB, sisi pendek di-resize, center crop, [0,1] float32.

    Ini pipeline evaluasi ultralytics sendiri (Resize -> CenterCrop -> ToTensor, mean 0 / std 1).
    """
    image = Image.open(path).convert("RGB")
    scale = size / min(image.size)
    image = image.resize(
        (max(size, round(image.width * scale)), max(size, round(image.height * scale))), Image.BILINEAR
    )
    left, top = (image.width - size) // 2, (image.height - size) // 2
    return np.asarray(image.crop((left, top, left + size, top + size)), dtype=np.float32) / 255.0


network = trained.model.eval()  # mode eval: kepala klasifikasi mengembalikan probabilitas, bukan logits


def torch_probabilities(pixels_hwc):
    """Jaringan PyTorch terlatih pada piksel yang sama — acuan yang harus disetujui hasil ekspor."""
    with torch.no_grad():
        output = network(torch.from_numpy(np.transpose(pixels_hwc, (2, 0, 1)))[None])
        probabilities = output[0] if isinstance(output, (tuple, list)) else output
        return probabilities[0].detach().cpu().numpy()


PROBE = load_pixels(sorted((DATASET_DIR / VAL_SPLIT / CLASSES[0]).iterdir())[0])

contract = inspect_model(TFLITE_PATH)
print(f"runtime       {contract.runtime}")
print(f"input         {contract.in_shape} {contract.in_dtype.__name__}")
print(f"output        {contract.out_shape} {contract.out_dtype.__name__}")
print(f"layout        {contract.layout} (aplikasi membaca bentuk tensor dan menyesuaikan diri)")
assert_app_contract(contract, len(CLASSES))

PROBE_SUM = float(predict(contract, PROBE).sum())
print(f"jumlah skor   {PROBE_SUM:.6f}")

if not 0.99 <= PROBE_SUM <= 1.01:
    print("graf mengembalikan logits, bukan probabilitas — mengekspor ulang dengan Softmax di ujungnya")

    for module in network.modules():
        if type(module).__name__ == "Classify":
            module.export = False  # di mode eval kepala lalu mengembalikan (probabilitas, logits); kita mau logits-nya


    class WithSoftmax(torch.nn.Module):
        """[1,3,224,224] RGB [0,1] -> [1,N] probabilitas softmax, yaitu kontrak aplikasi."""

        def __init__(self, network):
            super().__init__()
            self.network = network

        def forward(self, x):
            output = self.network(x)
            logits = output[1] if isinstance(output, (tuple, list)) else output
            return torch.nn.functional.softmax(logits, dim=1)


    from ultralytics.utils.export.litert import torch2litert

    target = TFLITE_PATH.with_name("garment_classifier_softmax.tflite")
    torch2litert(WithSoftmax(network), torch.zeros(1, 3, IMGSZ, IMGSZ), target, metadata={"task": "classify"})
    shutil.move(str(target), str(TFLITE_PATH))
    EXPORT_STEPS.append("torch2litert(WithSoftmax(best.pt), torch.zeros(1,3,224,224), ...) -> garment_classifier.tflite")

    contract = inspect_model(TFLITE_PATH)
    assert_app_contract(contract, len(CLASSES))
    PROBE_SUM = float(predict(contract, PROBE).sum())
    print(f"jumlah skor   {PROBE_SUM:.6f} (setelah ekspor ulang)")
    if not 0.99 <= PROBE_SUM <= 1.01:
        raise SystemExit("hasil ekspor tetap bukan distribusi probabilitas — jangan dipakai")

# Gerbang fidélitas: >= 100 foto held-out, piksel sama ke kedua model, bandingkan argmax-nya.
PER_CLASS = max(2, math.ceil(120 / len(CLASSES)))
held_out = []
for label in CLASSES:
    photos = sorted((DATASET_DIR / VAL_SPLIT / label).iterdir())
    step = max(1, len(photos) // PER_CLASS)
    held_out.extend((photo, label) for photo in photos[::step][:PER_CLASS])
if len(held_out) < 100:
    raise SystemExit(f"hanya {len(held_out)} foto held-out — pemeriksaan fidélitas butuh minimal 100")

rows = []
for photo, label in held_out:
    pixels = load_pixels(photo)
    expected = torch_probabilities(pixels)
    actual = predict(contract, pixels)
    rows.append(
        (
            label,
            CLASSES[int(expected.argmax())],
            CLASSES[int(actual.argmax())],
            float(actual.sum()),
            float(np.abs(expected - actual).max()),
        )
    )

IMAGE_COUNT = len(rows)
AGREE_COUNT = sum(1 for _, reference, converted, _, _ in rows if reference == converted)
TORCH_CORRECT = sum(1 for label, reference, _, _, _ in rows if label == reference)
TFLITE_CORRECT = sum(1 for label, _, converted, _, _ in rows if label == converted)
AGREEMENT = AGREE_COUNT / IMAGE_COUNT
TORCH_ACCURACY = TORCH_CORRECT / IMAGE_COUNT
TFLITE_ACCURACY = TFLITE_CORRECT / IMAGE_COUNT
MAX_SUM_ERROR = max(abs(row[3] - 1.0) for row in rows)
MAX_PROB_DIFF = max(row[4] for row in rows)
GATE_OK = TFLITE_ACCURACY >= 0.50 and TFLITE_ACCURACY >= TORCH_ACCURACY - 0.10

print(f"foto          {IMAGE_COUNT} held-out dari split {VAL_SPLIT}")
print(f"akurasi       pytorch {TORCH_ACCURACY:.4f}   .tflite {TFLITE_ACCURACY:.4f}")
print(f"kesesuaian    {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f} argmax identik")
print(f"maks |jml-1|  {MAX_SUM_ERROR:.3e}")
print(f"maks |beda|   {MAX_PROB_DIFF:.4f}")
print(f"gerbang       .tflite >= 0.50 dan >= pytorch - 0.10 -> {'LULUS' if GATE_OK else 'GAGAL'}")
if not GATE_OK:
    raise SystemExit(
        f"model hasil konversi mendapat {TFLITE_ACCURACY:.4f} sementara PyTorch {TORCH_ACCURACY:.4f} "
        "pada foto yang sama: konversinya rusak, jangan dipakai"
    )

## 7. Tulis `classes.json` dan `report.md`

`classes.json` memakai format repo kita — `{"version": 1, "source": ..., "labels": [...]}` — dan urutan `labels` **adalah** urutan output model. Karena nama folder kelas di langkah 3 sudah memakai ejaan `ml_workspace/labels.json`, ultralytics mengurutkannya secara abjad (kebetulan sama dengan urutan tabel di atas), dan `model.names` adalah urutan itu: indeks `i` di `labels` adalah arti skor `i`. `ml_workspace/scripts/gen_dart_labels.py` membaca file ini bersama `ml_workspace/labels.json` untuk menghasilkan `flutter_app/lib/logic/garment_labels.dart`; mengubah urutannya dengan tangan berarti setiap prediksi di aplikasi bergeser satu kelas.

`report.md` adalah jejak bukti: versi runtime, angka yang baru saja diukur (top-1, top-5, akurasi dan macro-F1 per kelas dari `sklearn`, hasil gerbang konversi, kesesuaian argmax), IO tensor hasil pembacaan verifier, sha256 dan ukuran `.tflite`, daftar perintah, catatan noise label, catatan lisensi, dan **perbandingan dengan baseline publik**.

Dua catatan yang **wajib** ikut terbawa ke laporan:

* **15 kelas tidak bisa dibandingkan langsung dengan model bootstrap 10 kelas (91,9 %).** Baseline publik untuk dataset ini adalah ViT-base fine-tune: **78,47 % akurasi / 0,7803 macro-F1** (30 epoch, split 40 %). Split kita 20 % (±100 foto/kelas vs 200 di baseline), jadi angka kita punya *support* lebih kecil — sebutkan keduanya, jangan hanya angka besarnya.
* **Label noise itu nyata di dataset ini.** Setidaknya enam berkas berbagi nama dasar UUID dengan dataset CC0 `alexeygrigorev/clothing-dataset`, dan dua di antaranya dipindahkan dari kelas berisik dataset itu (`Not sure`, `Other`) ke `Blazer`. Jadi confusion matrix adalah buktinya, bukan akurasi di judul.

In [ ]:
# Langkah 7/8 — tulis classes.json dan report.md.
import hashlib
import json
from datetime import datetime, timezone

CLASSES_JSON = ARTIFACTS_DIR / "classes.json"
CLASSES_JSON.write_text(
    json.dumps(
        {
            "version": 1,
            "source": (
                f"Ultralytics {MODEL} classification fine-tuned on ryanbadai/clothes-dataset "
                f"({len(CLASSES)} classes, imgsz {IMGSZ}, {EPOCHS} epochs, batch {BATCH}, seed {SEED_TRAIN}); "
                "index = the model's output index (ultralytics' alphabetical class-folder order)"
            ),
            "labels": CLASSES,
        },
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)
print(f"ditulis       {CLASSES_JSON} ({len(CLASSES)} label)")

TFLITE_BYTES = TFLITE_PATH.stat().st_size
TFLITE_SHA256 = hashlib.sha256(TFLITE_PATH.read_bytes()).hexdigest()

# Hoisted out of the report f-string: replacement fields must not carry the f-string's own quote.
DATE_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
GATE_LABEL = "PASSED" if GATE_OK else "FAILED"
GPU_NAME = torch.cuda.get_device_name(0) if GPU else "none (CPU)"
TRAIN_IMAGES = sum(counts["train"] for counts in COUNTS.values())
VAL_IMAGES = sum(counts["val"] for counts in COUNTS.values())
PER_CLASS_TABLE = "\n".join(
    f"| {label} | {REPORT_DICT[label]['precision']:.4f} | {REPORT_DICT[label]['recall']:.4f} | "
    f"{REPORT_DICT[label]['f1-score']:.4f} | {int(REPORT_DICT[label]['support'])} |"
    for label in CLASSES
)
LABEL_TABLE_MD = "\n".join(f"| {row['label']} | {row['category']} | {row['name']} |" for row in LABEL_TABLE)
COMMAND_STEPS = "\n".join(EXPORT_STEPS)
CLASS_LIST = ", ".join(CLASSES)

report = f"""# YOLOv8 classifier run — ryanbadai/clothes-dataset (15 Indonesian classes)

| | |
| --- | --- |
| date (UTC) | {DATE_STAMP} |
| notebook | `ml_workspace/yolo/kaggle_clothes_dataset.ipynb` (self-contained: no repository checkout) |
| dataset | `ryanbadai/clothes-dataset` at `{DATASET_ROOT}` |
| licence of the data | Kaggle tag MIT, description says the images were scraped from Carousell — coursework artifacts, not for redistribution (docs/adr/0007) |
| GPU | {GPU_NAME} |
| ultralytics / torch / python | {ultralytics.__version__} / {torch.__version__} / {sys.version.split()[0]} |
| model | `{MODEL}` (ImageNet-pretrained), fine-tuned head |
| data | {len(CLASSES)} classes, {TRAIN_IMAGES} train / {VAL_IMAGES} val images (stratified 80/20, seed {SEED}) |
| training | imgsz {IMGSZ}, {EPOCHS} epochs, batch {BATCH}, seed {SEED_TRAIN}, patience {PATIENCE}, device {DEVICE} |
| best weights | `{BEST_WEIGHTS}` |

## Measured

| metric | value |
| --- | --- |
| top-1 (`{VAL_SPLIT}` split, ultralytics pipeline) | {TOP1:.4f} |
| top-5 (`{VAL_SPLIT}` split, ultralytics pipeline) | {TOP5:.4f} |
| accuracy (sklearn, same split, second pass) | {ACCURACY:.4f} |
| macro-F1 (sklearn) | {MACRO_F1:.4f} |
| PyTorch top-1 on the {IMAGE_COUNT}-image fidelity subset | {TORCH_ACCURACY:.4f} |
| `.tflite` top-1 on the same subset | {TFLITE_ACCURACY:.4f} |
| argmax agreement, `.tflite` vs PyTorch | {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f} |
| largest `abs(sum(scores) - 1)` | {MAX_SUM_ERROR:.3e} |
| largest `abs(p_tflite - p_pytorch)` | {MAX_PROB_DIFF:.4f} |
| conversion gate (`.tflite` >= 0.50 and >= PyTorch - 0.10) | {GATE_LABEL} |
| confusion-matrix cells where ultralytics and sklearn disagree | {beda} |

Comparison, with the two caveats that matter:

* public baseline for this dataset: fine-tuned ViT-base-patch16-224-in21k, 30 epochs, 40 % stratified
  split -> **78.47 % accuracy / 0.7803 macro-F1**. This run used a 20 % split ({VAL_IMAGES} images,
  about {VAL_IMAGES // len(CLASSES)} per class against the baseline's 200), so the support is smaller
  and the numbers are only comparable in spirit.
* the shipped CC0 bootstrap model measured **91.9 % top-1 over 10 classes**: 15 classes are harder
  than 10, so neither number refutes the other. Never quote the 91.9 % next to this one without the
  class count in the same sentence.

## Known data problems (read before trusting the confusion matrix)

* **Label noise.** At least six files share UUID basenames with `alexeygrigorev/clothing-dataset`,
  and two of them were relabelled from that dataset's noisy classes (`Not sure`, `Other`) into
  `Blazer`. Expect `Blazer` and the other outerwear classes to absorb some of that noise; the
  confusion matrix, not the headline accuracy, is the evidence.
* **Three of the five Categories.** The 15 classes cover Tops, Bottoms and Outerwear only — there is
  no Shoes and no Accessories class, so a photographed shoe comes back with a garment label. The
  Correction step is the guard, and the class list is the real limit.
* **Split difference.** The baseline used a 40 % test split, this run 20 %; support per class differs.

## Exported model

* `{TFLITE_PATH}`
* size {TFLITE_BYTES:,} bytes, sha256 `{TFLITE_SHA256}`
* read by `{contract.runtime}`: input `{contract.in_shape}` {contract.in_dtype.__name__}, output `{contract.out_shape}` {contract.out_dtype.__name__}, layout {contract.layout}
* class order (`classes.json` `labels`, = the model's output indices): {CLASS_LIST}
* label table (must equal the 15 `ryanbadai/clothes-dataset` rows in `ml_workspace/labels.json`):

| label | Category | Nama |
| --- | --- | --- |
{LABEL_TABLE_MD}

## Commands

```bash
pip install -q ultralytics
# dataset attached as input: /kaggle/input/clothes-dataset/Clothes_Dataset
YOLO('{MODEL}').train(data={DATASET_DIR}, epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, seed={SEED_TRAIN}, project={RUNS_DIR})
YOLO('{BEST_WEIGHTS}').val(data={DATASET_DIR}, split='{VAL_SPLIT}', imgsz={IMGSZ})
{COMMAND_STEPS}
```

## Per class (`{VAL_SPLIT}` split)

| class | precision | recall | f1-score | support |
| --- | --- | --- | --- | --- |
{PER_CLASS_TABLE}

## Licence

Trained with Ultralytics YOLOv8 on `ryanbadai/clothes-dataset`. Ultralytics' licence page states
that models trained with their code are AGPL-3.0 by default and these weights are derivative work
of the `{MODEL}` checkpoint: shipping a build that contains this `.tflite` triggers AGPL-3.0's
source obligation, and this repository has no LICENSE file
(`docs/adr/0006-yolov8-classifier-agpl.md`). The dataset's Kaggle tag is MIT, but its description
says the images were scraped from Carousell, so the photographs are not ours to redistribute
(`docs/adr/0007-carousell-clothes-dataset.md`).
"""
(ARTIFACTS_DIR / "report.md").write_text(report, encoding="utf-8")
print(f"ditulis       {ARTIFACTS_DIR / 'report.md'}")

## 8. Bungkus dan unduh

Sesi Kaggle bisa mati, jadi langkah terakhir membungkus semua yang kamu butuhkan ke satu zip di **`/kaggle/working`** — folder yang tersimpan sebagai output notebook (maks 20 GB):

* `garment_classifier.tflite` — modelnya, dengan sha256 dan ukurannya dicetak di bawah;
* `classes.json` — urutan kelas yang dipakai model itu;
* `report.md` — angka terukur, perintah, catatan noise label dan lisensi;
* `classification_report.txt` / `.csv`, `per_class_accuracy.csv`, `confusion_matrix.csv`;
* `confusion_matrix.png` dan `confusion_matrix_normalized.png`.

Setelah zip ini ada di komputermu, langkah integrasi ke aplikasi ada di `ml_workspace/yolo/README.md` (salin `.tflite` ke `flutter_app/assets/models/garment_classifier.tflite`, lalu jalankan `gen_dart_labels.py --classes … --model …`).

In [ ]:
# Langkah 8/8 — bungkus artefak, cetak sha256/ukuran, dan tiga angka untuk dilaporkan.
import zipfile

ZIP_PATH = Path("/kaggle/working/gayaku_yolo_clothes.zip")
BUNDLE = (
    "garment_classifier.tflite",
    "classes.json",
    "report.md",
    "classification_report.txt",
    "classification_report.csv",
    "per_class_accuracy.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in BUNDLE:
        path = ARTIFACTS_DIR / name
        if path.is_file():
            archive.write(path, name)
        else:
            print(f"tidak ada, dilewati: {name}")

print(f"{TFLITE_PATH.name:<32}{TFLITE_BYTES:>12,} bytes")
print(f"{'sha256':<32}{TFLITE_SHA256}")
print(f"{ZIP_PATH.name:<32}{ZIP_PATH.stat().st_size:>12,} bytes")
print()
print("Tiga angka untuk dilaporkan:")
print(f"  1. top-1 ({VAL_SPLIT}, ultralytics) : {TOP1:.4f}")
print(f"  2. macro-F1 (sklearn)              : {MACRO_F1:.4f}")
print(f"  3. kesesuaian .tflite vs PyTorch   : {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f}")
print()
print("Selanjutnya: ml_workspace/yolo/README.md — bagian integrasi ke aplikasi.")

## Catatan penutup: lisensi, dan angka yang boleh dikutip

* **Bobotnya AGPL-3.0.** Semua model yang dilatih dengan kode Ultralytics berlisensi AGPL-3.0 menurut halaman lisensi mereka, dan `.tflite` ini adalah karya turunan dari checkpoint `yolov8n-cls.pt` — jadi mengedarkan APK yang memuatnya berarti menawarkan karya turunannya di bawah AGPL-3.0. Repo ini **belum punya berkas `LICENSE`**, jadi menambahkannya adalah prasyarat sebelum distribusi; `docs/adr/0006-yolov8-classifier-agpl.md` mencatat keputusan itu. Untuk kelas dan tugas kuliah, pemakaiannya bebas.
* **Datanya bukan milik kita.** Tag MIT di Kaggle menutupi labelnya; deskripsi dataset menyebut fotonya di-*scrape* dari Carousell. Sama seperti ADR-0003: bobot dan build yang memuatnya adalah artefak tugas, bukan untuk diedarkan (`docs/adr/0007-carousell-clothes-dataset.md`).
* **Hati-hati mengutip akurasi.** Baseline publik dataset ini 78,47 % / 0,7803 macro-F1 (ViT-base, split 40 %), sedangkan model bootstrap 10 kelas yang sekarang dikirim aplikasi mengukur 91,9 % top-1. Jumlah kelas dan ukuran split berbeda, jadi jangan pernah menyebut angka itu berdekatan tanpa menyebut kelas dan split-nya.